# About dada filtering

Try to find how `dada2` filtering performs with bacteria data:

In [ ]:
require("here")
require("ggplot2")
require("jsonlite")

Define some helper functions:

In [ ]:
# Define a function to calculate the filtered metrics
calculate_filtered_metrics <- function(data) {
    data$filtered_drop <- data$DADA2_input - data$filtered
    data$filtered_perc <- (data$filtered_drop / data$DADA2_input) * 100
    return(data)
}

# define a function to calculate merged metrics
calculate_merged_metrics <- function(data) {
    data$merged_drop <- data$filtered - data$merged
    data$merged_perc <- (data$merged_drop / data$filtered) * 100
    return(data)
}

# define a function to calculate chimera metrics
calculate_chimera_metrics <- function(data) {
    data$chimera_drop <- data$merged - data$nonchim
    data$chimera_perc <- (data$chimera_drop / data$merged) * 100
    return(data)
}

In [ ]:
# Define a function to read parameters file
read_latest_json <- function(
    directory, 
    pattern="params_\\d{4}-\\d{2}-\\d{2}_\\d{2}-\\d{2}\\-\\d{2}.json",
    columns_of_interest = c("trunc_qmin", "trunc_rmin", "trunclenf", "trunclenr", "max_ee")) {
  
  # List all files in the directory that match the pattern
  files <- list.files(directory, pattern = pattern, full.names = TRUE)
  
  # Check if any files were found
  if (length(files) == 0) {
    print(directory)
    print(files)
    stop("No files matching the pattern were found.")
  }
  
  # Sort the files in decreasing order (latest first)
  sorted_files <- sort(files, decreasing = TRUE)
  
  # The latest file is the first one in the sorted list
  latest_file <- sorted_files[1]
  
  # Read the latest JSON file
  json_data <- fromJSON(latest_file)
  
  # Filter the data based on the columns of interest and create a named list
  filtered_data <- setNames(lapply(columns_of_interest, function(col) json_data[[col]]), columns_of_interest)
  
  return(filtered_data)
}

In [ ]:
# define an helper function that, starting from a directory, reads the latest JSON file and returns the parameters
# and the metrics caming from dada2
read_results <- function(directory) {
    # Read the latest JSON file
    params <- read_latest_json(here(directory, "pipeline_info"))
    
    # Read the metrics file
    metrics <- read.csv(here::here(directory, "dada2", "DADA2_stats.tsv"), sep = "\t", header = TRUE)
    
    # Calculate the filtered metrics
    metrics <- calculate_filtered_metrics(metrics)
    
    # Calculate the merged metrics
    metrics <- calculate_merged_metrics(metrics)
    
    # Calculate the chimera metrics
    metrics <- calculate_chimera_metrics(metrics)
    
    return(list(params = params, metrics = metrics))
}

Try to increase trimming size:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 225,
    "trunclenr": 205,
    "max_ee": 6
}
```

In [ ]:
R2_205ee6_result = read_results("results-bacteria")
R2_205ee6 <- R2_205ee6_result$metrics
R2_205ee6_result$params

In [ ]:
summary(R2_205ee6)

Another test with a different `ee`:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 210,
    "max_ee": 6,
}
```

In [ ]:
R2_210ee6_result = read_results("results-bacteria.1")
R2_210ee6 <- R2_210ee6_result$metrics
R2_210ee6_result$params

In [ ]:
summary(R2_210ee6)

Another test with a different `ee`:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 210,
    "max_ee": 5,
}
```

In [ ]:
R2_210ee5_result = read_results("results-bacteria.2")
R2_210ee5 <- R2_210ee5_result$metrics
R2_210ee5_result$params

In [ ]:
summary(R2_210ee5)

More or less, I've figure out where to truncate reads. This is an attempt to discover what changes
by increasing `ee`.

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 210,
    "max_ee": 4
}
```

In [ ]:
R2_210ee4_result = read_results("results-bacteria.3")
R2_210ee4 <- R2_210ee4_result$metrics
R2_210ee4_result$params


In [ ]:
summary(R2_210ee4)

Read data using a more stringent parameters on R2:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 210,
    "max_ee": 3
}
```

In [ ]:
R2_210ee3_results <- read_results("results-bacteria.4")
R2_210ee3 <- R2_210ee3_results$metrics
R2_210ee3_results$params

In [ ]:
summary(R2_210ee3)

Read data using another less permissive parameters on R2:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 220,
    "max_ee": 3
}
```

In [ ]:
R2_220ee3_results <- read_results("results-bacteria.5")
R2_220ee3 <- R2_220ee3_results$metrics
R2_220ee3_results$params

In [ ]:
summary(R2_220ee3)

Read data calculated by setting read size to 230bp length:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 230,
    "max_ee": 2
}
```

This was the analysis I want to call after determining reads length with `figaro`,
however, the parameters it suggested were less stringent and I didn'r recover any
sequences (maybe because I've removed all the sequences below 250bp?). So, I choose
to lower `trunclenf` and `trunclenr` parameters.

In [ ]:
fixed_length_results <- read_results("results-bacteria.6")
fixed_length <- fixed_length_results$metrics
fixed_length_results$params

In [ ]:
summary(fixed_length)

In this run, I've tried to increase the min quality used to determine where to
truncate forward and reverse reads:

```json
{
    "trunc_qmin": 30,
    "trunc_rmin": 0.75,
    "trunclenf": null,
    "trunclenr": null,
    "max_ee": 2
}
```

In [ ]:
minq30_results <- read_results("results-bacteria.7")
minq30 <- minq30_results$metrics
minq30_results$params

In [ ]:
summary(minq30)

Read data generated using default parameters (determine read length automatically):

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": null,
    "trunclenr": null,
    "max_ee": 2
}
```

In [ ]:
default_params_results <- read_results("results-bacteria.8")
default_params <- default_params_results$metrics
default_params_results$params

In [ ]:
summary(default_params)

In [ ]:
# Set the plot size
options(repr.plot.width = 12, repr.plot.height = 6)

# Combine the data into a single dataframe
combined_data <- rbind(
    data.frame(sample = default_params$sample, filtered_perc = default_params$filtered_perc, method = "Default"),
    data.frame(sample = minq30$sample, filtered_perc = minq30$filtered_perc, method = "minq30"),
    data.frame(sample = fixed_length$sample, filtered_perc = fixed_length$filtered_perc, method = "Fixed230"),
    data.frame(sample = R2_220ee3$sample, filtered_perc = R2_220ee3$filtered_perc, method = "R2_220ee3"),
    data.frame(sample = R2_210ee3$sample, filtered_perc = R2_210ee3$filtered_perc, method = "R2_210ee3"),
    data.frame(sample = R2_210ee4$sample, filtered_perc = R2_210ee4$filtered_perc, method = "R2_210ee4"),
    data.frame(sample = R2_210ee5$sample, filtered_perc = R2_210ee5$filtered_perc, method = "R2_210ee5"),
    data.frame(sample = R2_210ee6$sample, filtered_perc = R2_210ee6$filtered_perc, method = "R2_210ee6"),
    data.frame(sample = R2_205ee6$sample, filtered_perc = R2_205ee6$filtered_perc, method = "R2_205ee6")
)

# order data series
combined_data$method <- factor(combined_data$method, levels = c(
    "Default",
    "minq30",
    "Fixed230",
    "R2_220ee3",
    "R2_210ee3",
    "R2_210ee4",
    "R2_210ee5",
    "R2_210ee6",
    "R2_205ee6"
))

# Plot the barplot
ggplot(combined_data, aes(x = sample, y = filtered_perc, fill = method)) +
    geom_bar(stat = "identity", position = "dodge") +
    labs(title = "Drop in Reads After Filtering by Method", x = "Sample", y = "Percentage of Reads Lost") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

Incrementing only the *minimum quality* to 30 has no significant effect on the
number of reads discarded. Decrementing the *read length* to 230bp, however,
seems to save more reads. By fixing the R2 length to 220bp, and increasing the 
expected errors to 3 saves a lot of reads, even in samples which seems to have 
a low quality.

In [ ]:
# Set the plot size
options(repr.plot.width = 12, repr.plot.height = 6)

# Combine the data into a single dataframe
combined_data <- rbind(
    data.frame(sample = default_params$sample, merged_perc = default_params$merged_perc, method = "Default"),
    data.frame(sample = minq30$sample, merged_perc = minq30$merged_perc, method = "minq30"),
    data.frame(sample = fixed_length$sample, merged_perc = fixed_length$merged_perc, method = "Fixed230"),
    data.frame(sample = R2_220ee3$sample, merged_perc = R2_220ee3$merged_perc, method = "R2_220ee3"),
    data.frame(sample = R2_210ee3$sample, merged_perc = R2_210ee3$merged_perc, method = "R2_210ee3"),
    data.frame(sample = R2_210ee4$sample, merged_perc = R2_210ee4$merged_perc, method = "R2_210ee4"),
    data.frame(sample = R2_210ee5$sample, merged_perc = R2_210ee5$merged_perc, method = "R2_210ee5"),
    data.frame(sample = R2_210ee6$sample, merged_perc = R2_210ee6$merged_perc, method = "R2_210ee6"),
    data.frame(sample = R2_205ee6$sample, merged_perc = R2_205ee6$merged_perc, method = "R2_205ee6")
)

# order data series
combined_data$method <- factor(combined_data$method, levels = c(
    "Default",
    "minq30",
    "Fixed230",
    "R2_220ee3",
    "R2_210ee3",
    "R2_210ee4",
    "R2_210ee5",
    "R2_210ee6",
    "R2_205ee6"
))

# Plot the barplot
ggplot(combined_data, aes(x = sample, y = merged_perc, fill = method)) +
    geom_bar(stat = "identity", position = "dodge") +
    labs(title = "Drop in Reads After Merging by Method", x = "Sample", y = "Percentage of Reads Lost") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
# Set the plot size
options(repr.plot.width = 12, repr.plot.height = 6)

# Combine the data into a single dataframe
combined_data <- rbind(
    data.frame(sample = default_params$sample, chimera_perc = default_params$chimera_perc, method = "Default"),
    data.frame(sample = minq30$sample, chimera_perc = minq30$chimera_perc, method = "minq30"),
    data.frame(sample = fixed_length$sample, chimera_perc = fixed_length$chimera_perc, method = "Fixed230"),
    data.frame(sample = R2_220ee3$sample, chimera_perc = R2_220ee3$chimera_perc, method = "R2_220ee3"),
    data.frame(sample = R2_210ee3$sample, chimera_perc = R2_210ee3$chimera_perc, method = "R2_210ee3"),
    data.frame(sample = R2_210ee4$sample, chimera_perc = R2_210ee4$chimera_perc, method = "R2_210ee4"),
    data.frame(sample = R2_210ee5$sample, chimera_perc = R2_210ee5$chimera_perc, method = "R2_210ee5"),
    data.frame(sample = R2_210ee6$sample, chimera_perc = R2_210ee6$chimera_perc, method = "R2_210ee6"),
    data.frame(sample = R2_205ee6$sample, chimera_perc = R2_205ee6$chimera_perc, method = "R2_205ee6")
)

#  order data series
combined_data$method <- factor(combined_data$method, levels = c(
    "Default",
    "minq30",
    "Fixed230",
    "R2_220ee3",
    "R2_210ee3",
    "R2_210ee4",
    "R2_210ee5",
    "R2_210ee6",
    "R2_205ee6"
))

# Plot the barplot
ggplot(combined_data, aes(x = sample, y = chimera_perc, fill = method)) +
    geom_bar(stat = "identity", position = "dodge") +
    labs(title = "Drop in Reads After Chimera detection by Method", x = "Sample", y = "Percentage of Reads Lost") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))